1. Framework

In [ ]:
import torch
import pandas as pd
import numpy as np

from textblob import TextBlob
import emoji
import re

import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.metrics import classification_report, accuracy_score
from sentence_transformers import SentenceTransformer

2. Data Load

In [ ]:
#Dataset Definition
twits_25k_path = '25k_dataset.csv'
twits_32k_path = 'hatespeech_train.csv'
#twits_150_path = ''
twits_25k_preprocessed_path = 'twits_25k_preprocessed.csv'

twits_25k = pd.read_csv(twits_25k_path)
twits_32k = pd.read_csv(twits_32k_path)
#twits_120 = pd.read_csv(twits_150_path)
twits_25k_preprocessed = pd.read_csv(twits_25k_preprocessed_path)

In [ ]:
twits_25k = twits_25k[["id","class","tweet"]]
twits_25k = twits_25k.rename(columns={"class":"label"})

twits_25k_preprocessed = twits_25k_preprocessed[["id","label","tweet"]]
# twits_25k_preprocessed = twits_25k_preprocessed.rename(columns={"class":"label"})

In [ ]:
twits_25k_preprocessed.head()

In [ ]:

original_dataset = twits_25k_preprocessed

# Especificar el tamaño del subconjunto que deseas
subset_size = 2000  # ajusta este valor según tus necesidades

# Obtener un subconjunto aleatorio
random_subset_25k = original_dataset.sample(n=subset_size, random_state=42)

# Ahora, random_subset contiene un subconjunto aleatorio de tu conjunto de datos


In [ ]:
random_subset_25k.shape

In [ ]:
def preprocess_dataset(dataset):
    # Change text to lowercase
    dataset['tweet'] = dataset['tweet'].str.lower()
    # Eliminate URLs
    dataset['tweet'] = dataset['tweet'].replace(r'http\S+', '', regex=True).replace(r'www\S+', '', regex=True)
    # Use TextBlob for spelling correction
    dataset['tweet'] = dataset['tweet'].apply(lambda x: str(TextBlob(x).correct()))
    # Convert emojis to text and keep sentiment
    dataset['tweet'] = dataset['tweet'].apply(lambda x: emoji.demojize(x))
    # Keep hashtags
    dataset['tweet'] = dataset['tweet'].apply(lambda x: re.sub(r'#', '', x))
    # Eliminate @mentions
    dataset['tweet'] = dataset['tweet'].replace(r'@\S+', '', regex=True)
    return dataset

In [ ]:
# Uso de la función con tu ruta de archivo
twits_25k_preprocessed = preprocess_dataset(twits_25k)

# Muestra las primeras filas del DataFrame resultante
print(twits_25k_preprocessed.head())

In [ ]:
twits_25k_preprocessed.to_csv("twits_25k_preprocessed.csv")

3. Prediction Number 1: DistilBERT

In [ ]:
# Cargar el conjunto de datos

test_df = twits_25k_preprocessed

# Contar la cantidad de muestras por clase
class_counts = test_df['label'].value_counts()
# Determinar la cantidad mínima de muestras entre las clases
min_class_count = class_counts.min()
# Realizar downscaling para balancear las clases
balanced_df = pd.DataFrame()
for label in test_df['label'].unique():
    label_subset = test_df[test_df['label'] == label].sample(min_class_count, random_state=42)
    balanced_df = pd.concat([balanced_df, label_subset])

# Mezclar el dataframe resultante
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Guardar el nuevo conjunto de datos balanceado
balanced_dataset_path = 'twits_25k_balanced.csv'
balanced_df.to_csv(balanced_dataset_path, index=False)

test_df = balanced_df

# Representaciones vectoriales de clases

class_representations = {
    '0': "This is an example of hate speech",
    '1': "This is an example of offensive language",
    '2': "This is an example that does not belong to any of the above categories."
}
# Model of text representation
model = SentenceTransformer('distilbert-base-nli-mean-tokens')

# Get repr vector for each class
class_vectors = {label: model.encode([text])[0] for label, text in class_representations.items()}

# Función para predecir la clase basada en la similitud de coseno
def predict_class(text):
    text_vector = model.encode([text])[0]
    similarities = {label: np.dot(text_vector, class_vectors[label]) / (np.linalg.norm(text_vector) * np.linalg.norm(class_vectors[label])) for label in class_vectors}
    predicted_class = max(similarities, key=similarities.get)
    return predicted_class

# Realizar predicciones en el conjunto de prueba
test_df['predicted_class'] = test_df['tweet'].apply(predict_class)

# Convertir etiquetas numéricas a cadenas
test_df['label'] = test_df['label'].astype(str)
test_df['predicted_class'] = test_df['predicted_class'].astype(str)

# Calcular métricas de clasificación
accuracy = accuracy_score(test_df['label'], test_df['predicted_class'])
classification_report_str = classification_report(test_df['label'], test_df['predicted_class'])

# Mostrar resultados
print(f'Accuracy: {accuracy:.4f}')
print('Classification Report:')
print(classification_report_str)



In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, auc
# Calcular la curva ROC (Variante adaptada para clasificación por similitud)
fpr, tpr, _ = roc_curve(test_df['label'].apply(lambda x: 1 if x == '0' else 0), test_df['predicted_class'].apply(lambda x: 1 if x == '0' else 0))
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
conf_matrix = confusion_matrix(test_df['label'], test_df['predicted_class'])
print(conf_matrix)

In [ ]:

# Cargar el conjunto de datos

test_df = random_subset_25k

# Separar datos de entrenamiento y prueba
#train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


# Representaciones vectoriales de clases
class_representations = {
    '0': "This is an example of hate speech",
    '1': "This is an example of offensive language",
    '2': "This is an example that does not belong to any of the above categories."
}

# Modelo de representación de texto RoBERTa
model_name = 'roberta-base'
tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name)

# Obtener representaciones vectoriales para cada clase
class_vectors = {label: tokenizer(text, return_tensors='pt')['input_ids'] for label, text in class_representations.items()}

# Función para predecir la clase basada en la similitud de coseno
def predict_class(text):
    input_ids = tokenizer(text, return_tensors='pt')['input_ids']
    with torch.no_grad():
        logits = model(input_ids)['logits']
    similarities = {label: np.dot(logits.flatten().numpy(), class_vectors[label].flatten().numpy()) / (np.linalg.norm(logits) * np.linalg.norm(class_vectors[label])) for label in class_vectors}
    predicted_class = max(similarities, key=similarities.get)
    return predicted_class


# Realizar predicciones en el conjunto de prueba
test_df['predicted_class'] = test_df['tweet'].apply(predict_class)

# Convertir etiquetas numéricas a cadenas
test_df['label'] = test_df['label'].astype(str)
test_df['predicted_class'] = test_df['predicted_class'].astype(str)

# Calcular métricas de clasificación
accuracy = accuracy_score(test_df['label'], test_df['predicted_class'])
classification_report_str = classification_report(test_df['label'], test_df['predicted_class'])

# Mostrar resultados
print(f'Accuracy: {accuracy:.4f}')
print('Classification Report:')
print(classification_report_str)


In [ ]:
random_subset_25k['roberta'] = all_predictions_roberta

In [ ]:
random_subset_25k

In [ ]:
# Agregar las predicciones al DataFrame
twits_25k_preprocessed['predicted_label'] = predictions


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Obtener las etiquetas reales del conjunto de datos
true_labels = twits_25k_preprocessed['label'].values

# Calcular la precisión
accuracy = accuracy_score(true_labels, predictions)
print(f'Accuracy on Dataset: {accuracy:.4f}')

# Mostrar un informe de clasificación
print('Classification Report:')
print(classification_report(true_labels, predictions))

In [ ]:
import os

# Ruta de la carpeta
folder_path = r'C:\Users\PabloFabianFaundezGa\Desktop\Hate Speech'

# Lista para almacenar los nombres de los archivos
file_names = []

# Recorrer los archivos en la carpeta
for file in os.listdir(folder_path):
    # Añadir el nombre del archivo a la lista
    file_names.append(file)

# Imprimir la lista de nombres de archivos
print(file_names)
